# 11 · SHAP e interpretabilidad del modelo final (Persona B)

**Pregunta:** ¿por qué el modelo da las probabilidades que da? ¿Qué variables pesan más y cuánto aporta cada una
frente al Elo?

**Qué es un valor SHAP.** Para un partido, reparte la predicción entre las variables: cada una recibe una cantidad
que dice cuánto movió el resultado respecto de un partido "promedio" (el *fondo*). Positivo = empuja hacia esa
clase, negativo = la aleja. Valor base + suma de los SHAP = puntaje del modelo para ese partido.

**Por qué aquí es casi gratis.** El modelo final es una logística multinomial sobre variables imputadas y
estandarizadas (`z`). Para un modelo lineal, el SHAP de la variable *i* hacia la clase *k* es exactamente

$$\text{SHAP}_{k,i} = \beta_{k,i}\,\bigl(z_i - \overline{z_i}^{\,\text{fondo}}\bigr).$$

Si el fondo son los partidos con que se ajustó el escalador (train), $\overline{z_i} = 0$ y el SHAP es
$\beta_{k,i} \, z_i$: lo mismo que ya calcula `ModeloFinal.factores()` y que muestra la app.
Este notebook lo comprueba con la librería `shap` y produce las figuras.

**Reglas:** el modelo se entrena **solo con train** (2017-18 a 2022-23) y se explican los 380 partidos de
**validación** (2023-24). El test 2024-25 no se toca. Los SHAP están en escala *logit* (el puntaje lineal antes del
softmax), no en puntos de probabilidad.

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import shap

from src.config import CLASSES
from src.splits import split
from src.models.final import FEATURES_FINAL, ModeloFinal
from src.models.logistic import NOMBRES_ES, tune_logistic, coef_table
from src.evaluation.interpretabilidad import (explicador_shap, valores_shap, valor_base,
                                              shap_clase_predicha, importancia_global)
from src.evaluation.figuras import fig_shap_global, fig_shap_ejemplo

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
train, val = split(df)                                  # sin use_test: el test no se toca
assert train["date"].max() < val["date"].min()

busqueda = tune_logistic(train, FEATURES_FINAL)         # C con TimeSeriesSplit dentro de train
modelo = ModeloFinal(busqueda.best_estimator_, FEATURES_FINAL)
explicador = explicador_shap(modelo, train)             # fondo = train imputado y escalado (completo)
sv = valores_shap(modelo, explicador, val)              # (380, 10, 3), clases en orden H, D, A
probs = modelo.predict_proba(val)
print(f"train: {len(train)} | val: {len(val)} | C = {busqueda.best_params_['logistica__C']:.4f}")
print("forma de los SHAP:", sv.shape)

## 1. ¿Coinciden los SHAP con `factores()`?

Se compara, partido por partido, el SHAP hacia la clase más probable con el `impacto` que devuelve `factores()`.
También se mira qué pasa con el fondo **por defecto** de `shap`: si se le pasa la matriz sin más, `LinearExplainer`
la envuelve en `maskers.Independent(max_samples=100)` y usa **100 partidos al azar**. Su media ya no es 0 y los
valores se desplazan.

In [ ]:
pred = shap_clase_predicha(modelo, sv, val)
a_col = {v: k for k, v in NOMBRES_ES.items()}
dif = []
for i in range(len(val)):
    fac = modelo.factores(val.iloc[[i]], top=len(FEATURES_FINAL))
    dif.append(max(abs(pred.iloc[i][a_col[f["variable"]]] - f["impacto"]) for f in fac))
print(f"Fondo completo (train)    -> diferencia máxima con factores(): {max(dif):.1e}")
print(f"Media del fondo completo  -> máx |media de z|: {np.abs(explicador.mean).max():.1e}")

Z_train = modelo.pipeline[:-1].transform(train[FEATURES_FINAL])
defecto = shap.LinearExplainer(modelo.pipeline.named_steps["logistica"], Z_train)
print(f"Fondo por defecto (100)   -> máx |media de z|: {np.abs(defecto.mean).max():.3f}")
sv_def = shap_clase_predicha(modelo, valores_shap(modelo, defecto, val), val)
print(f"Fondo por defecto (100)   -> diferencia máxima con factores(): {(sv_def - pred).abs().to_numpy().max():.3f}")

# Valor base + suma de SHAP = puntaje lineal de cada clase
logit = modelo.pipeline.named_steps["logistica"]
puntaje = logit.decision_function(modelo.pipeline[:-1].transform(val[FEATURES_FINAL]))
orden = [list(logit.classes_).index(c) for c in CLASSES]
print("base + suma de SHAP = puntaje:", np.allclose(valor_base(modelo, explicador) + sv.sum(axis=1), puntaje[:, orden]))

## 2. Importancia global: media de |SHAP| por variable

Para cada partido se toma el SHAP hacia **su** resultado predicho (lo mismo que muestra la app) y se promedia el
valor absoluto en los 380 partidos. También se muestra la media de |SHAP| por clase.

In [ ]:
imp = importancia_global(pred)
por_clase = pd.DataFrame({c: np.abs(sv[:, :, j]).mean(axis=0) for j, c in enumerate(CLASSES)},
                         index=FEATURES_FINAL)
tabla_imp = pd.DataFrame({"|SHAP| clase predicha": imp,
                          "% del Elo": imp / imp["elo_diff"]}).join(por_clase.add_prefix("|SHAP| "))
tabla_imp.index = [NOMBRES_ES[c] for c in tabla_imp.index]
tabla_imp.round(3)

In [ ]:
print("Clase predicha en validación:",
      pd.Series(probs.argmax(axis=1)).map(dict(enumerate(CLASSES))).value_counts().to_dict())
resto = imp.drop("elo_diff")
print(f"Elo: {imp['elo_diff']:.3f} | suma de las otras 9: {resto.sum():.3f} ({resto.sum() / imp['elo_diff']:.0%} del Elo)")
fig = fig_shap_global(imp, "../docs/fig_shap_global.png", n_partidos=len(val))

### Efecto neto por grupo de variables

Las 9 variables que no son Elo suman más que el Elo si se miran una por una, pero **se cancelan entre sí**
(por ejemplo, puntos y goles del mismo equipo suelen empujar en sentidos opuestos, ver sección 3). Para saber
cuánto aporta cada grupo de verdad, se suman primero los SHAP del grupo en cada partido y luego se promedia el
valor absoluto. La última columna cuenta en qué % de los partidos el grupo empuja hacia el mismo lado que el Elo.

In [ ]:
grupos = {"Elo": ["elo_diff"],
          "Forma (6 variables)": [c for c in FEATURES_FINAL if "form" in c],
          "Descanso (3 variables)": [c for c in FEATURES_FINAL if "rest" in c]}
neto = pd.DataFrame({g: pred[cols].sum(axis=1) for g, cols in grupos.items()})
tabla_grupos = pd.DataFrame({
    "media |Σ SHAP|": neto.abs().mean(),
    "% del Elo": neto.abs().mean() / neto["Elo"].abs().mean(),
    "suma de |SHAP| individuales": [imp[cols].sum() for cols in grupos.values()],
    "% partidos mismo signo que Elo": [(np.sign(neto[g]) == np.sign(neto["Elo"])).mean() for g in neto],
})
tabla_grupos.round(3)

## 3. ¿Por qué los puntos por partido tienen el signo "al revés"?

En la tabla de coeficientes, **más puntos del local en los últimos 5 bajan la probabilidad de que gane el local**
(y más puntos del visitante la suben). No es un error de orden de columnas: es **colinealidad**. Los puntos están muy
correlacionados con los goles a favor/en contra y con el Elo, que ya están en el modelo. Con todo eso fijo, lo que
queda de los puntos es "puntos de más respecto de lo que dicen los goles y el Elo", que se parece a **suerte** y
tiende a no repetirse. El efecto es pequeño (C bajo = mucha regularización).

In [ ]:
print(coef_table(modelo.pipeline, FEATURES_FINAL).round(3))
corr = train[["home_form_pts", "home_form_gf", "home_form_ga", "elo_diff"]].corr().round(2)
corr.index = corr.columns = [NOMBRES_ES[c] for c in corr.columns]
corr

## 4. Un partido explicado: Arsenal – Sheffield Utd (28 de octubre de 2023)

Arsenal (subcampeón de la temporada anterior) de local ante el Sheffield United, recién ascendido.
Un favorito clarísimo. Terminó **5–0**.

In [ ]:
fila = val[(val["home"] == "Arsenal") & (val["away"] == "Sheffield Utd")]
assert len(fila) == 1
i = val.index.get_loc(fila.index[0])
p = probs[i]
print(fila[["date", "home", "away", "home_goals", "away_goals", "result"]].to_string(index=False))
print("Probabilidades H, D, A:", p.round(3), "-> predice", CLASSES[p.argmax()])

explicacion = pd.DataFrame({"valor": fila[FEATURES_FINAL].iloc[0], "SHAP": pred.iloc[i]})
explicacion = explicacion.reindex(explicacion["SHAP"].abs().sort_values(ascending=False).index)
explicacion.index = [NOMBRES_ES[c] for c in explicacion.index]
print(f"Valor base (H): {valor_base(modelo, explicador)[0]:.3f} | suma de SHAP: {pred.iloc[i].sum():.3f}")
explicacion.round(3)

In [ ]:
fig = fig_shap_ejemplo(pred.iloc[i], fila[FEATURES_FINAL].iloc[0], p,
                       titulo="Arsenal 5–0 Sheffield Utd · 28 de octubre de 2023",
                       resultado_real="ganó el local (5–0)",
                       ruta="../docs/fig_shap_ejemplo.png", top=6)

## 5. Conclusión

| Grupo | media \|SHAP\| (neto, por partido) | Frente al Elo |
|---|---|---|
| **Diferencia de Elo** | **0.438** | 100 % |
| Forma de los últimos 5 (6 variables) | 0.088 | **20 %** |
| Días de descanso (3 variables) | 0.012 | **3 %** |

- **`factores()` ya es SHAP.** Con el fondo = train completo, la diferencia máxima entre `shap.LinearExplainer` y
  `factores()` en los 380 partidos es ~1e-16 (cero numérico): la media del fondo escalado es 0, así que
  SHAP = coeficiente × valor escalado. El único cuidado es **no** usar el fondo por defecto de `shap`
  (100 partidos al azar): su media llega a 0.23 desviaciones y desplaza los valores hasta 0.04.
  La app usa el modelo entrenado con train+val y su escalador está ajustado con esos mismos partidos, así que
  lo que muestra la app es exactamente el SHAP con fondo = train+val.
- **El Elo manda.** Es, por mucho, la variable que más mueve la predicción (0.44 de media). La siguiente,
  puntos por partido del visitante, pesa un 22 % de lo que pesa el Elo, y las demás de forma entre 7 % y 19 %.
- **La forma aporta en conjunto un 20 % de lo que aporta el Elo**, no el ~100 % que sugiere sumar las barras.
  Sus 6 variables se compensan entre sí (puntos contra goles del mismo equipo) y solo en el 67 % de los
  partidos empujan hacia el mismo lado que el Elo: matizan la predicción, casi nunca la dan vuelta.
- **El descanso es casi irrelevante** (3 % del Elo). Está en el modelo porque la decisión del 18-10 fue que
  influyera en la app, no porque cambie las probabilidades.
- **Signos "al revés" en los puntos por partido:** efecto de la colinealidad con goles y Elo (sección 3). Para
  la presentación conviene hablar de **"forma"** como grupo y no interpretar cada variable de forma por separado.
- **El modelo nunca predice empate como resultado más probable** (235 partidos H, 145 A, 0 D), así que las
  explicaciones de la app siempre son hacia "gana el local" o "gana el visitante".
- **Partido de ejemplo:** Arsenal–Sheffield Utd (87 % local, terminó 5–0). El Elo aporta +1.24; los 3.4 goles
  recibidos por partido del Sheffield, +0.34. Valor base (0.32) + suma de SHAP (1.52) = puntaje 1.84 de "gana el
  local", que tras el softmax da el 87 %.
- **Limitaciones:** los SHAP explican **al modelo**, no causas del fútbol; están en escala logit (no en puntos de
  probabilidad); y con variables correlacionadas el reparto entre ellas depende de cuáles estén en el modelo.
